In [5]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

In [15]:
#Specialised reducer
from langgraph.graph.message import add_messages

In [16]:
from typing import Literal, TypedDict, Optional, Annotated

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [43]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7, max_output_tokens=512)

In [44]:
def chat_with_ai(state: ChatState):
    messages = state['messages']
    prompt_template = PromptTemplate(
        input_variables=["messages"],
        template="You are a helpful assistant. Continue the conversation based on the following messages. Reply in short-oneline if possi:\n\n{messages}"
    )
    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({"messages" : messages})
    return  {"messages": [AIMessage(content=response)]}
    

In [45]:
graph = StateGraph(ChatState)

In [46]:
graph.add_node('chat_node' , chat_with_ai)

#add_edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [47]:
workflow = graph.compile()

In [48]:
input_state = {
    "messages" : [
        SystemMessage(content="You are a Pre-historic animals expert. Be precise and concise in your answers."),
        HumanMessage(content="What is the largest dinosaur that ever lived?")
    ]
}

In [49]:
final_state = workflow.invoke(input_state)

In [50]:
for message in final_state['messages']:
    print(f"{message.type}: {message.content}")

system: You are a Pre-historic animals expert. Be precise and concise in your answers.
human: What is the largest dinosaur that ever lived?
ai: The largest known dinosaur is *Argentinosaurus*, weighing around 70-100 metric tons.


In [53]:
chat_history = [SystemMessage(content="You are a helpful assistant. Be precise and concise in your answers. Whenever possible, reply in short one-line answers.")]

In [54]:
while True:
    input_text = input()
    print(f"User: {input_text}")
    list_of_words = input_text.strip().lower().split(" ")
    end_words = ["exit", "quit", "bye"]
    if any(word in list_of_words for word in end_words):
        print("Exiting the chat. Goodbye!")
        break

    chat_history.append(HumanMessage(content=input_text))

    final_state = workflow.invoke({"messages": chat_history})

    print(f"AI: {final_state['messages'][-1].content}")


User: Hello I am Harshit, call me by name in your each following response
AI: Hello Harshit, how can I help you today?
User: Tell me the strongest dinosaur ever lived\
AI: Hello Harshit, Spinosaurus is widely considered the largest and among the strongest predatory dinosaurs to have ever lived.
User: was it bigger than elephant
AI: Yes Harshit, *Spinosaurus* and *Tyrannosaurus rex* were both much larger than an elephant.
User: what about the vegeterian ones
AI: Hello Harshit, the largest herbivorous dinosaur was *Argentinosaurus*, which was vastly larger and heavier than any elephant.
User: and was that bigger then SauroPosiedons
AI: Hello Harshit, no, *Sauroposeidon* was actually taller and likely heavier than *Tyrannosaurus rex*.
User: nArgentinoSaurus vs suroposideons
AI: Hello Harshit, Argentinosaurus was generally heavier and more massive, while Sauroposeidon may have been slightly taller.
User: ok cool bye
Exiting the chat. Goodbye!
